# 1. LOAD DATASET

In [ ]:
# Bitcoin Dataset Validation - Check for Missing Minutes
# Dataset: 3.1M records, 1-minute intervals, 2017-2023

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("="*60)
print("BITCOIN DATASET VALIDATION")
print("Full column analysis")
print("="*60)

# Load data (use sample for GitHub)
df = pd.read_csv('/kaggle/input/bitcoin-price-dataset/bitcoin_2017_to_2023.csv',nrows=1000000)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nCOLUMN BREAKDOWN:")
print(df.dtypes)

# 2. Check Column Quality

In [ ]:
# Check for missing values
print("\n" + "="*50)
print("MISSING VALUES CHECK")
print("="*50)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_pct
})
print(missing_report[missing_report['missing_count'] > 0])

In [ ]:
# Check for zero/negative values
print("\n" + "="*50)
print("DATA QUALITY CHECK")
print("="*50)

issues = []

# Price columns should be positive
for col in ['open', 'high', 'low', 'close']:
    if col in df.columns:
        negative = (df[col] <= 0).sum()
        if negative > 0:
            issues.append(f"{col}: {negative} non-positive values")

# Volume columns should be non-negative
for col in ['volume', 'quote_asset_volume', 'number_of_trades',
            'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume']:
    if col in df.columns:
        negative = (df[col] < 0).sum()
        if negative > 0:
            issues.append(f" {col}: {negative} negative values")

if issues:
    for issue in issues:
        print(issue)
else:
    print("All columns pass basic quality checks")

# 3. Validate OHLC Logic

In [ ]:
# Check that high >= low, close within range
print("\n" + "="*50)
print("OHLC INTEGRITY CHECK")
print("="*50)

invalid_high_low = (df['high'] < df['low']).sum()
invalid_close_range = ((df['close'] < df['low']) | (df['close'] > df['high'])).sum()

print(f"high < low violations: {invalid_high_low}")
print(f"close outside [low, high]: {invalid_close_range}")

if invalid_high_low == 0 and invalid_close_range == 0:
    print("All OHLC relationships are valid!")

# 4. Order flow analysis

In [ ]:
# Order flow analysis
print("\n" + "="*50)
print("ORDER FLOW ANALYSIS (Quant Feature)")
print("="*50)

# Calculate taker sell volume
df['taker_sell_base_asset_volume'] = df['volume'] - df['taker_buy_base_asset_volume']

# Order flow imbalance (-1 to 1 scale)
df['ofi'] = (df['taker_buy_base_asset_volume'] - df['taker_sell_base_asset_volume']) / df['volume']

print(f"Order Flow Imbalance (OFI) Statistics:")
print(f"   Mean: {df['ofi'].mean():.4f}")
print(f"   Std: {df['ofi'].std():.4f}")
print(f"   Min: {df['ofi'].min():.4f}")
print(f"   Max: {df['ofi'].max():.4f}")

# When is buying pressure highest?
high_buy = df[df['ofi'] > 0.5]
print(f"\nPeriods with strong buying pressure (OFI > 0.5): {len(high_buy):,} rows")

In [ ]:
# Trade intensity
print("\n" + "="*50)
print("TRADE ACTIVITY ANALYSIS")
print("="*50)

print(f" Number of Trades Statistics:")
print(f"   Mean per minute: {df['number_of_trades'].mean():.1f}")
print(f"   Max per minute: {df['number_of_trades'].max():.0f}")
print(f"   Min per minute: {df['number_of_trades'].min():.0f}")

# Identify high-activity periods
high_activity = df[df['number_of_trades'] > df['number_of_trades'].quantile(0.99)]
print(f"\n Top 1% high-activity minutes: {len(high_activity):,} rows")

# 5. Create Summary Table

In [ ]:
# Summary statistics for all columns
print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)

summary = df[['open', 'high', 'low', 'close', 'volume','number_of_trades', 'ofi']].describe()

print(summary)

# 6. Validation Report

In [ ]:
# Create final report
validation_report = {
    'total_rows': len(df),
    'date_range': f"{df['timestamp'].min()} to {df['timestamp'].max()}",
    'missing_values': df.isnull().sum().sum(),
    'ohlc_valid': invalid_high_low == 0 and invalid_close_range == 0,
    'ofi_mean': df['ofi'].mean(),
    'avg_trades_per_minute': df['number_of_trades'].mean(),
    'columns_available': df.columns.tolist()
}

print("\n" + "="*50)
print("VALIDATION REPORT")
print("="*50)
for key, value in validation_report.items():
    print(f"{key}: {value}")